In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Module 2: Exploratory Data Analysis (EDA)

Before building any model, you must understand your data. EDA is the process of
examining datasets to summarise their main characteristics, spot problems, and
uncover patterns.

## Section 2.1 — Handling Missing Data

**Objectives**

By the end of this section you will be able to:

- Detect and quantify missing values in a dataset.
- Choose an appropriate imputation strategy for numeric and categorical columns.
- Apply `fillna()` and `dropna()` to produce a clean dataset.
- Assess the potential impact of missing data on model quality.

> **Cooking analogy:** Imagine receiving a grocery delivery where several items
> are missing — no eggs, no butter. You have a choice: send the whole delivery
> back (drop the rows), substitute with a similar ingredient (impute with the
> median or mode), or redesign the recipe to work around the gap. Cooking blindly
> with missing ingredients ruins the dish. The same is true for your data.

Real-world business data is almost always incomplete. Learning how to detect
and handle missing values is a fundamental skill.

In [ ]:
import pandas as pd
import numpy as np

# Simulate a customer dataset with missing values
np.random.seed(42)

n = 200
data = {
    "CustomerID"  : range(1001, 1001 + n),
    "Age"         : np.where(np.random.rand(n) < 0.08, np.nan,
                             np.random.randint(22, 70, n).astype(float)),
    "Income"      : np.where(np.random.rand(n) < 0.12, np.nan,
                             np.random.normal(65_000, 20_000, n).round(-2)),
    "Purchases"   : np.random.randint(1, 50, n),
    "Segment"     : np.where(np.random.rand(n) < 0.05, np.nan,
                             np.random.choice(["Bronze","Silver","Gold"], n)),
}

df = pd.DataFrame(data)
print("Dataset shape:", df.shape)
print(df.head())

### Detecting Missing Values

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
pct_missing = (missing / len(df) * 100).round(1)

missing_report = pd.DataFrame({
    "Missing_Count"  : missing,
    "Missing_Pct_%"  : pct_missing
})
print(missing_report[missing_report["Missing_Count"] > 0])

In [ ]:
# Visualise missingness pattern
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(missing_report.index, missing_report["Missing_Pct_%"], color="steelblue")
ax.set_title("Percentage of Missing Values per Column")
ax.set_ylabel("Missing (%)")
ax.set_xlabel("Column")
plt.tight_layout()
plt.show()

### Strategies for Handling Missing Data

| Strategy | When to Use |
|---|---|
| **Drop rows** | Very few rows affected and data is large |
| **Fill with mean/median** | Numerical columns, missing at random |
| **Fill with mode** | Categorical columns |
| **Forward/backward fill** | Time-series data |
| **Predictive imputation** | Advanced; missing not at random |

In [ ]:
df_clean = df.copy()

# 1. Fill numeric columns with median (robust to outliers)
df_clean["Age"]    = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean["Income"] = df_clean["Income"].fillna(df_clean["Income"].median())

# 2. Fill categorical column with mode (most frequent value)
df_clean["Segment"] = df_clean["Segment"].fillna(df_clean["Segment"].mode()[0])

# Verify no missing values remain
print("Missing after cleaning:", df_clean.isnull().sum().sum())
print("\nMedian Age used for imputation:", df["Age"].median())
print(f"Median Income used:            ${df['Income'].median():,.0f}")

In [ ]:
# Alternative: drop rows with any missing values (use when data is abundant)
df_dropped = df.dropna()
print(f"Rows before: {len(df)}, after dropping NAs: {len(df_dropped)}")

---

### Student Task 2.1

Run the cell below to create a sales dataset with missing values. Then:

1. Report which columns have missing values and what percentage is missing.
2. Choose an appropriate strategy for each column and justify your choice.
3. Apply your chosen strategy to produce a clean dataset `df_sales_clean`.
4. Verify the clean dataset has zero missing values.

In [ ]:
# Dataset provided — do not change this cell
np.random.seed(7)
m = 150
df_sales = pd.DataFrame({
    "OrderID"    : range(5001, 5001 + m),
    "Region"     : np.where(np.random.rand(m) < 0.06, np.nan,
                            np.random.choice(["North","South","East","West"], m)),
    "Sales"      : np.where(np.random.rand(m) < 0.10, np.nan,
                            np.random.uniform(500, 50_000, m).round(2)),
    "Quantity"   : np.random.randint(1, 100, m),
    "Discount"   : np.where(np.random.rand(m) < 0.15, np.nan,
                            np.random.uniform(0, 0.4, m).round(2)),
})

# Your cleaning code here

---

### Evaluation Questions 2.1

1. Which method returns a Boolean DataFrame showing where values are missing?
   a) `df.missing()`
   b) `df.isna()` (correct)
   c) `df.nullcheck()`
   d) `df.find_nan()`

2. When is replacing missing values with the **median** preferred over the mean?
   a) When there are no outliers
   b) When the data is perfectly symmetric
   c) When outliers are present in the column (correct)
   d) When the column contains text

3. Filling missing values using values from the previous row is called:
   a) Backward fill
   b) Mean imputation
   c) Forward fill (correct)
   d) Random imputation

4. If 40% of values in a column are missing, which action is most appropriate?
   a) Fill with the mean — always safe
   b) Drop all rows with missing values
   c) Investigate why data is missing and consider dropping the column (correct)
   d) Replace with zero

5. `df.dropna()` removes:
   a) Columns with missing values
   b) Only rows where all values are NaN
   c) Any row that contains at least one missing value (correct)
   d) Zero values

---